# Criacao da camada Bronze

## Configuracao do ambiente

In [13]:
import os
from dotenv import find_dotenv, load_dotenv

# Procura o .env subindo as pastas a partir de onde o notebook está rodando.
load_dotenv(find_dotenv(usecwd=True), override=True)

def variavel(nome, padrao=None):
    """Lê uma variável de ambiente. Valor vazio conta como 'não definida'."""
    valor = os.getenv(nome) or padrao
    if valor is None:
        raise RuntimeError(f"Variável de ambiente {nome} não definida. Veja o .env.")
    return valor

PROJECT_ID = variavel("PROJECT_ID")  # projeto onde as consultas rodam e a Bronze é gravada
SOURCE_DATASET = variavel("SOURCE_DATASET", "basedosdados.br_inep_avaliacao_alfabetizacao")
BRONZE_DATASET = variavel("BRONZE_DATASET", "bronze")
BQ_LOCATION = os.getenv("BQ_LOCATION") or None  # opcional: se vazio, usa a localização da fonte
DESTINO = os.getenv("DESTINO") or "local"  # opcional: se vazio, grava localmente

In [14]:
def adicionar_metadados_ingestao(df, nome_tabela):
    """Adiciona colunas de metadados de ingestão ao DataFrame."""
    from datetime import datetime
    import pandas as pd
    import uuid
    
    df["id_ingestao"] = uuid.uuid4()
    df["data_ingestao"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df["tabela_origem"] = nome_tabela

    return df

In [15]:
from pathlib import Path

BRONZE_DIR = Path(os.getenv("BRONZE_DIR") or Path.cwd().parent / "data" / "bronze")

def gravar_bronze(df, nome_tabela):
    """Grava o DataFrame na Bronze sem sobrescrever rodadas anteriores."""
    if DESTINO == "local":
        id_ingestao = df["id_ingestao"].iloc[0]
        pasta = BRONZE_DIR / nome_tabela
        pasta.mkdir(parents=True, exist_ok=True)
        caminho = pasta / f"id_ingestao={id_ingestao}.parquet"
        if caminho.exists():
            raise FileExistsError(f"Rodada já gravada: {caminho}")
        df.to_parquet(caminho, index=False)
        return caminho
    raise ValueError(f"DESTINO inválido ou não implementado: {DESTINO}")

##  Conectando com bigquery

In [16]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

def carregar_tabela_bq(client, nome_tabela, dataset=SOURCE_DATASET):
    """Carrega uma tabela do BigQuery em um DataFrame do Pandas."""
    from google.cloud import bigquery

    query = f"SELECT * FROM `{dataset}.{nome_tabela}`"
    df = client.query(query).to_dataframe(create_bqstorage_client=True)

    return df

/Users/fagnertonon/Github/fiap-ciencia-de-dados/FASE 2/tech-challenge-fase-2/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
import pandas as pd

TABELAS = ["alunos", "municipio", "uf", "meta_alfabetizacao_brasil",
           "meta_alfabetizacao_uf", "meta_alfabetizacao_municipio", "dicionario"]

dfs = {}
for nome in TABELAS:
    print(f"Carregando tabela {nome}...")
    df = carregar_tabela_bq(client, nome)
    print(f"Tabela {nome} carregada com {df.shape[0]} linhas e {df.shape[1]} colunas.")
    dfs[nome] = adicionar_metadados_ingestao(df, nome)
    print(f"Tabela {nome} com metadados de ingestão adicionados: {dfs[nome].shape[0]} linhas e {dfs[nome].shape[1]} colunas.")
    
    print(f"Gravando tabela {nome} na Bronze...")
    gravar_bronze(dfs[nome], nome)
    print(f"Tabela {nome} gravada na Bronze com sucesso.")


Carregando tabela alunos...
Tabela alunos carregada com 3867999 linhas e 12 colunas.
Tabela alunos com metadados de ingestão adicionados: 3867999 linhas e 15 colunas.
Gravando tabela alunos na Bronze...
Tabela alunos gravada na Bronze com sucesso.
Carregando tabela municipio...
Tabela municipio carregada com 23995 linhas e 15 colunas.
Tabela municipio com metadados de ingestão adicionados: 23995 linhas e 18 colunas.
Gravando tabela municipio na Bronze...
Tabela municipio gravada na Bronze com sucesso.
Carregando tabela uf...
Tabela uf carregada com 145 linhas e 15 colunas.
Tabela uf com metadados de ingestão adicionados: 145 linhas e 18 colunas.
Gravando tabela uf na Bronze...
Tabela uf gravada na Bronze com sucesso.
Carregando tabela meta_alfabetizacao_brasil...
Tabela meta_alfabetizacao_brasil carregada com 3 linhas e 11 colunas.
Tabela meta_alfabetizacao_brasil com metadados de ingestão adicionados: 3 linhas e 14 colunas.
Gravando tabela meta_alfabetizacao_brasil na Bronze...
Tabela